<a href="https://colab.research.google.com/github/marikhakhishvili/Facial-Expression-Recognition-Challenge/blob/main/model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 8.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
! mkdir ~/.kaggle

In [4]:
!cp /content/drive/MyDrive/cs231n/assignments/assignment4/kaggle.json ~/.kaggle/kaggle.json

In [5]:
! chmod 600 ~/.kaggle/kaggle.json

download competition data

In [6]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:01<00:00, 223MB/s]



In [7]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [8]:
import wandb
!wandb login --relogin

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


check if there are any problems with the data


In [9]:
"""
Sanity check for FER2013 pipeline.

1. Builds the smallest possible model: 1 conv layer + FC.
2. Checks initial loss is close to -ln(1/7) ~= 1.9459 (random baseline, 7 classes).
3. Trains on a fixed ~20-image subset and checks loss -> ~0 (model can memorize).
4. Logs everything to Wandb as run_0_sanity_check.

Adjust CSV_PATH / column names to match your actual train.csv (some FER2013
versions have an "emotion"+"pixels" only train.csv, others also have "Usage").
"""

import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb

CSV_PATH = "train.csv"   # update to your file path
NUM_SAMPLES = 20
NUM_CLASSES = 7
NUM_STEPS = 300
LR = 1e-2
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)


class TinyFERSubset(Dataset):
    def __init__(self, csv_path, n_samples=20):
        df = pd.read_csv(csv_path)

        # If a "Usage" column exists, restrict to the training split
        if "Usage" in df.columns:
            df = df[df["Usage"] == "Training"]

        df = df.reset_index(drop=True).iloc[:n_samples]

        images = []
        for pixel_str in df["pixels"]:
            arr = np.array(pixel_str.split(), dtype=np.float32).reshape(48, 48)
            images.append(arr)

        self.images = np.stack(images) / 255.0   # (N, 48, 48), normalized
        self.labels = df["emotion"].values.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx]).unsqueeze(0).float()  # (1, 48, 48)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label


class TinyNet(nn.Module):
    """Smallest reasonable model: 1 conv layer + FC."""

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)  # 48 -> 24
        self.fc = nn.Linear(8 * 24 * 24, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = x.flatten(1)
        return self.fc(x)


def run_sanity_check():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    wandb.init(
        project="fer2013-assignment",
        group="sanity_checks",
        name="run_0_sanity_check",
        config={
            "model": "TinyNet (1 conv + FC)",
            "num_samples": NUM_SAMPLES,
            "lr": LR,
            "num_steps": NUM_STEPS,
        },
    )

    dataset = TinyFERSubset(CSV_PATH, n_samples=NUM_SAMPLES)
    loader = DataLoader(dataset, batch_size=NUM_SAMPLES, shuffle=True)

    model = TinyNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    images, labels = next(iter(loader))
    images, labels = images.to(device), labels.to(device)

    # --- Check 1: initial loss vs random baseline ---
    model.eval()
    with torch.no_grad():
        initial_loss = criterion(model(images), labels).item()

    expected_random_loss = -math.log(1.0 / NUM_CLASSES)
    print(f"Initial loss: {initial_loss:.4f}  (expected ~{expected_random_loss:.4f})")

    wandb.log({
        "sanity/initial_loss": initial_loss,
        "sanity/expected_random_loss": expected_random_loss,
    }, step=0)

    # --- Check 2: overfit the tiny subset ---
    model.train()
    for step in range(1, NUM_STEPS + 1):
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        acc = (logits.argmax(dim=1) == labels).float().mean().item()
        wandb.log({"sanity/loss": loss.item(), "sanity/accuracy": acc}, step=step)

        if step % 50 == 0 or step == 1:
            print(f"Step {step:4d} | loss {loss.item():.4f} | acc {acc:.2f}")

    final_loss, final_acc = loss.item(), acc
    print("\n--- Summary ---")
    print(f"Initial loss: {initial_loss:.4f} vs expected {expected_random_loss:.4f}")
    print(f"Final loss after {NUM_STEPS} steps: {final_loss:.4f}, accuracy: {final_acc:.2f}")

    if final_loss < 0.05 and final_acc > 0.95:
        print("PASS: model memorized the subset -> forward/backward pass is correct.")
    else:
        print("WARNING: did not fully overfit. Check data loading, labels, LR, or model.")

    wandb.summary.update({
        "initial_loss": initial_loss,
        "expected_random_loss": expected_random_loss,
        "final_loss": final_loss,
        "final_accuracy": final_acc,
    })
    wandb.finish()


if __name__ == "__main__":
    run_sanity_check()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mkhak23 (mkhak23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Initial loss: 2.0955  (expected ~1.9459)
Step    1 | loss 2.0955 | acc 0.00
Step   50 | loss 0.0006 | acc 1.00
Step  100 | loss 0.0001 | acc 1.00
Step  150 | loss 0.0001 | acc 1.00
Step  200 | loss 0.0001 | acc 1.00
Step  250 | loss 0.0001 | acc 1.00
Step  300 | loss 0.0000 | acc 1.00

--- Summary ---
Initial loss: 2.0955 vs expected 1.9459
Final loss after 300 steps: 0.0000, accuracy: 1.00
PASS: model memorized the subset -> forward/backward pass is correct.


sanity/accuracy,▁▄██████████████████████████████████████
sanity/expected_random_loss,▁
sanity/initial_loss,▁
sanity/loss,█▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
expected_random_loss,1.94591
final_accuracy,1
final_loss,5e-05
initial_loss,2.0955
sanity/accuracy,1
sanity/expected_random_loss,1.94591
sanity/initial_loss,2.0955


Preprocessing

In [10]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb
df = pd.read_csv('./train.csv')
def pixels_to_image_array(pixels_str):
    """
    Converts a space-separated pixel string into a normalized
    1 x 48 x 48 NumPy array suitable for PyTorch.
    """

    # Convert string to NumPy array of floats
    pixels = np.array(pixels_str.split(), dtype=np.float32)

    # Reshape into a 48x48 image
    image = pixels.reshape(48, 48)

    # Normalize pixel values from [0, 255] to [0, 1]
    image = image / 255.0

    # Add channel dimension: (48, 48) -> (1, 48, 48)
    image = np.expand_dims(image, axis=0)

    return image

# Apply preprocessing to every image
df["pixels"] = df["pixels"].apply(pixels_to_image_array)

# Check the shape of the first image
print(df["pixels"].iloc[0].shape)
# Expected output: (1, 48, 48)

(1, 48, 48)


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

X = np.stack(df["pixels"].values)   # shape: (N, 1, 48, 48)
y = df["emotion"].values

# First split: 85% for (train+val), 15% for test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Second split: From the 85%, split into ~70% train and ~15% val
# Calculate the test_size for the second split: (0.15 / (0.70 + 0.15)) = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(0.15 / 0.85), random_state=42, stratify=y_temp
)

In [12]:
print("\nEmotion distribution in New Training Set:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nEmotion distribution in Validation Set:")
print(pd.Series(y_val).value_counts(normalize=True))

print("\nEmotion distribution in Test Set:")
print(pd.Series(y_test).value_counts(normalize=True))


Emotion distribution in New Training Set:
3    0.251356
6    0.172929
4    0.168201
2    0.142672
0    0.139189
5    0.110425
1    0.015228
Name: proportion, dtype: float64

Emotion distribution in Validation Set:
3    0.251219
6    0.172974
4    0.168331
2    0.142791
0    0.139076
5    0.110518
1    0.015092
Name: proportion, dtype: float64

Emotion distribution in Test Set:
3    0.251219
6    0.172974
4    0.168331
2    0.142791
0    0.139076
5    0.110518
1    0.015092
Name: proportion, dtype: float64


In [13]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Model

In [14]:
class CNN_Model2(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [15]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return total_loss / len(loader), correct / total

Training

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model2().to(device)

criterion = nn.CrossEntropyLoss()

In [17]:
wandb.init(
    project="fer2013-assignment",
    name="model2_deepercnn_lr5e-4_bs64",
    config={
        "model": "CNN_Model2",
        "lr": 5e-4,
        "batch_size": 64,
        "epochs": 25,
        "optimizer": "Adam",
        "notes": "deeper CNN + batchnorm"
    }
)

config = wandb.config

In [18]:
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

In [19]:
EPOCHS = 25

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

Epoch 1/25
Train Loss: 1.5751, Train Acc: 0.3851
Val Loss: 1.3786, Val Acc: 0.4748
Epoch 2/25
Train Loss: 1.3185, Train Acc: 0.5011
Val Loss: 1.3010, Val Acc: 0.4997
Epoch 3/25
Train Loss: 1.2072, Train Acc: 0.5418
Val Loss: 1.2284, Val Acc: 0.5268
Epoch 4/25
Train Loss: 1.1207, Train Acc: 0.5763
Val Loss: 1.2058, Val Acc: 0.5377
Epoch 5/25
Train Loss: 1.0429, Train Acc: 0.6077
Val Loss: 1.1830, Val Acc: 0.5535
Epoch 6/25
Train Loss: 0.9484, Train Acc: 0.6460
Val Loss: 1.1770, Val Acc: 0.5609
Epoch 7/25
Train Loss: 0.8643, Train Acc: 0.6747
Val Loss: 1.3069, Val Acc: 0.5238
Epoch 8/25
Train Loss: 0.7728, Train Acc: 0.7151
Val Loss: 1.1859, Val Acc: 0.5733
Epoch 9/25
Train Loss: 0.6750, Train Acc: 0.7538
Val Loss: 1.2426, Val Acc: 0.5623
Epoch 10/25
Train Loss: 0.5802, Train Acc: 0.7865
Val Loss: 1.3866, Val Acc: 0.5531
Epoch 11/25
Train Loss: 0.4839, Train Acc: 0.8269
Val Loss: 1.4418, Val Acc: 0.5417
Epoch 12/25
Train Loss: 0.3864, Train Acc: 0.8685
Val Loss: 1.4545, Val Acc: 0.5630
E

In [20]:
wandb.finish()

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train_acc,▁▂▃▃▄▄▄▅▅▆▆▇▇▇▇██████████
train_loss,█▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▅▅▇▇▄█▇▇▆▇▆▇▆▆▇▄▇▆▇▆▅▆▆
val_loss,▂▂▁▁▁▁▂▁▁▂▂▂▃▃▄▅▅▆▆▆▆▆█▇▇
epoch,25
train_acc,0.98721
train_loss,0.0555
val_acc,0.54469
val_loss,2.62098


Training with smaller learning rate and less epochs

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model2().to(device)

criterion = nn.CrossEntropyLoss()

In [22]:
wandb.init(
    project="fer2013-assignment",
    name="model2_deepercnn_lr1e-4_bs64",
    config={
        "model": "CNN_Model2",
        "lr": 1e-4,
        "batch_size": 64,
        "epochs": 10,
        "optimizer": "Adam",
        "notes": "deeper CNN + batchnorm"
    }
)

config = wandb.config

In [23]:
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

In [24]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

Epoch 1/10
Train Loss: 1.6127, Train Acc: 0.3701
Val Loss: 1.4832, Val Acc: 0.4349
Epoch 2/10
Train Loss: 1.4139, Train Acc: 0.4656
Val Loss: 1.3717, Val Acc: 0.4799
Epoch 3/10
Train Loss: 1.3047, Train Acc: 0.5082
Val Loss: 1.3109, Val Acc: 0.5006
Epoch 4/10
Train Loss: 1.2253, Train Acc: 0.5421
Val Loss: 1.3196, Val Acc: 0.4925
Epoch 5/10
Train Loss: 1.1522, Train Acc: 0.5750
Val Loss: 1.3103, Val Acc: 0.5020
Epoch 6/10
Train Loss: 1.0845, Train Acc: 0.6006
Val Loss: 1.2493, Val Acc: 0.5275
Epoch 7/10
Train Loss: 1.0192, Train Acc: 0.6299
Val Loss: 1.2657, Val Acc: 0.5254
Epoch 8/10
Train Loss: 0.9508, Train Acc: 0.6580
Val Loss: 1.2445, Val Acc: 0.5359
Epoch 9/10
Train Loss: 0.8937, Train Acc: 0.6807
Val Loss: 1.2501, Val Acc: 0.5319
Epoch 10/10
Train Loss: 0.8149, Train Acc: 0.7152
Val Loss: 1.2633, Val Acc: 0.5333


In [25]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▄▄▅▆▆▇▇█
train_loss,█▆▅▅▄▃▃▂▂▁
val_acc,▁▄▆▅▆▇▇███
val_loss,█▅▃▃▃▁▂▁▁▂
epoch,10
train_acc,0.71515
train_loss,0.8149
val_acc,0.53332
val_loss,1.26328


training with lr 5e-4, epochs 10

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model2().to(device)

criterion = nn.CrossEntropyLoss()

In [27]:
wandb.init(
    project="fer2013-assignment",
    name="model2_deepercnn_lr5e-4_bs64_epochs10",
    config={
        "model": "CNN_Model2",
        "lr": 5e-4,
        "batch_size": 64,
        "epochs": 10,
        "optimizer": "Adam",
        "notes": "deeper CNN + batchnorm"
    }
)

config = wandb.config

In [28]:
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

In [29]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

Epoch 1/10
Train Loss: 1.5686, Train Acc: 0.3913
Val Loss: 1.4321, Val Acc: 0.4335
Epoch 2/10
Train Loss: 1.3326, Train Acc: 0.4905
Val Loss: 1.3469, Val Acc: 0.4869
Epoch 3/10
Train Loss: 1.2150, Train Acc: 0.5376
Val Loss: 1.2344, Val Acc: 0.5212
Epoch 4/10
Train Loss: 1.1248, Train Acc: 0.5759
Val Loss: 1.2222, Val Acc: 0.5349
Epoch 5/10
Train Loss: 1.0436, Train Acc: 0.6086
Val Loss: 1.1445, Val Acc: 0.5603
Epoch 6/10
Train Loss: 0.9634, Train Acc: 0.6428
Val Loss: 1.1628, Val Acc: 0.5647
Epoch 7/10
Train Loss: 0.8748, Train Acc: 0.6736
Val Loss: 1.1714, Val Acc: 0.5593
Epoch 8/10
Train Loss: 0.7900, Train Acc: 0.7103
Val Loss: 1.1958, Val Acc: 0.5716
Epoch 9/10
Train Loss: 0.7024, Train Acc: 0.7409
Val Loss: 1.2256, Val Acc: 0.5656
Epoch 10/10
Train Loss: 0.5925, Train Acc: 0.7870
Val Loss: 1.2392, Val Acc: 0.5728


### Evaluate on Test Set

In [30]:
test_loss, test_acc = evaluate(model, test_loader)

print(f"\nFinal Test Loss: {test_loss:.4f}, Final Test Acc: {test_acc:.4f}")

wandb.log({
    "test_loss": test_loss,
    "test_acc": test_acc
})


Final Test Loss: 1.2246, Final Test Acc: 0.5883


In [31]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
test_acc,▁
test_loss,▁
train_acc,▁▃▄▄▅▅▆▇▇█
train_loss,█▆▅▅▄▄▃▂▂▁
val_acc,▁▄▅▆▇█▇███
val_loss,█▆▃▃▁▁▂▂▃▃
epoch,10
test_acc,0.58834
test_loss,1.22459
train_acc,0.78701
